# Paper 2 — Goal 1
## Notebook 10B: FINAL completion, sensitivities, canonical outputs, and figure-review package

This notebook is intentionally separated from Notebook 10.

**Why a new notebook is safer:** Notebook 10 now contains the long historical permutation diagnostics/recovery sequence as well as the original all-in-one runner. Re-running that runner would invoke obsolete permutation logic and its old `PERMUTATION_REPEATS=10` seal condition. Notebook 10B instead treats the completed Notebook-10 model and permutation checkpoints as governed inputs, resumes only genuinely missing sensitivity models, and builds the corrected final reporting layer.

### What this notebook does

1. imports the exact validated Goal-1 modeling implementation from the uploaded/frozen Notebook 10 **without running `run_goal1()`**;
2. validates/reloads the 10-repeat primary OOF model ladder;
3. reuses valid 2,000-bootstrap tables or recomputes them only if needed;
4. hard-audits the already-final corrected 1,000-draw diagnosis null;
5. **finalizes the newly completed 1,000-draw severity null** using the matched 3-repeat statistic;
6. writes the explicit permutation-method implementation amendment:
   - primary model performance remains 5 folds × 10 repeats;
   - formal diagnosis and severity permutation statistics use matched 5 folds × 3 repeats;
   - diagnosis stratification is regenerated after each diagnosis-label permutation;
   - severity retains the fixed master folds because the continuous severity outcome did not construct those folds;
7. runs/resumes the complete prespecified Goal-1 sensitivity set and computes 2,000 participant-cluster bootstrap CIs;
8. assembles the canonical Goal-1 OOF, metrics, permutation, split, sensitivity, calibration, and descriptive tables;
9. generates Figure 2 and supplementary figure candidates in a dedicated review folder;
10. writes a computational completion manifest with status `PASS_PENDING_FIGURE_REVIEW`.

### Important

This notebook **does not rerun either 1,000-permutation null**.

It also does not write the final publication freeze yet. We will visually audit the generated figures first, exactly as we did for Goal 2, then write the small final Goal-1 figure/freeze notebook.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import sys
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

COMPLETION_ENGINE = "goal1-finalize-and-figure-review-v1.0.0"
FINAL_PRIMARY_REPEATS = 10
FINAL_BOOTSTRAPS = 2000
FINAL_PERMUTATIONS = 1000
FINAL_PERMUTATION_REPEATS = 3

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def atomic_json(payload, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("." + path.name + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def atomic_csv(frame, path: Path, *, allow_empty=False):
    path = Path(path)
    if frame.empty and not allow_empty:
        raise ValueError(f"Refusing to write empty table: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("." + path.name + ".tmp")
    frame.to_csv(tmp, index=False)
    if not tmp.exists() or tmp.stat().st_size == 0:
        raise IOError(f"Temporary CSV is empty: {tmp}")
    os.replace(tmp, path)

def find_root():
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        p = Path(override).expanduser().resolve()
        if (p / "outputs" / "goal1").exists() and (p / "data" / "processed").exists():
            return p
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {p}")

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "outputs" / "goal1").exists() and (p / "data" / "processed").exists():
            return p
    raise FileNotFoundError(
        "Could not locate Paper_2_Leakage/Code. Run this notebook from the repository "
        "or set PAPER2_ROOT."
    )

ROOT = find_root()

SOURCE_NOTEBOOK_CANDIDATES = [
    "10_goal1_primary_analysis_final_v1_1_2000_bootstraps.ipynb",
    "10_goal1_primary_analysis_development_v1_1_300_bootstraps.ipynb",
]

def locate_source_notebook():
    search_roots = [Path.cwd(), ROOT, ROOT / "notebooks"]
    for name in SOURCE_NOTEBOOK_CANDIDATES:
        for base in search_roots:
            p = base / name
            if p.exists():
                return p.resolve()
        matches = list(ROOT.rglob(name))
        if matches:
            return sorted(matches, key=lambda p: (len(p.parts), str(p)))[0].resolve()
    raise FileNotFoundError(
        "Could not locate the authoritative Notebook 10. Expected one of: "
        + ", ".join(SOURCE_NOTEBOOK_CANDIDATES)
    )

SOURCE_NOTEBOOK = locate_source_notebook()
SOURCE_NOTEBOOK_SHA256 = sha256_file(SOURCE_NOTEBOOK)

print("Project root:", ROOT)
print("Source Notebook 10:", SOURCE_NOTEBOOK)
print("Source Notebook SHA-256:", SOURCE_NOTEBOOK_SHA256)
print("Completion engine:", COMPLETION_ENGINE)


Project root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Source Notebook 10: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\notebooks\10_goal1_COMPLETE_REBUILT_v1_1_final2000BS.ipynb
Source Notebook SHA-256: e1bf319c7088b09456d53a89fd0825563bc6540d0069ca2045c888d0e2d872ee
Completion engine: goal1-finalize-and-figure-review-v1.0.0


## 1. Import Notebook 10's exact modeling implementation without running the all-in-one pipeline

Only Notebook-10 code cells through the definition of `run_goal1()` are executed. The terminal statement `GOAL1 = run_goal1()` is removed before execution.

This gives Notebook 10B the exact same:

- frozen populations;
- Core-Q / Extended-Q definitions;
- QCHAN reconstruction;
- preprocessing;
- nested-CV engine;
- checkpoint validation;
- bootstrap functions;
- sensitivity specifications;
- figure-generation functions.

Notebook-10's later permutation-recovery cells are **not executed**.

In [2]:
source_nb = json.loads(SOURCE_NOTEBOOK.read_text(encoding="utf-8"))

goal1_ns = {
    "__name__": "__goal1_notebook10_import__",
    "__file__": str(SOURCE_NOTEBOOK),
}

executed_cells = 0
runner_definition_seen = False

for cell_index, cell in enumerate(source_nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    if not source.strip():
        continue

    # We need the original helpers + run_goal1 definition, but never its invocation.
    if "def run_goal1():" in source:
        source = re.sub(
            r"\n\s*GOAL1\s*=\s*run_goal1\(\)\s*$",
            "\n",
            source,
            flags=re.S,
        )
        exec(
            compile(source, f"{SOURCE_NOTEBOOK.name}::<cell {cell_index}>", "exec"),
            goal1_ns,
            goal1_ns,
        )
        executed_cells += 1
        runner_definition_seen = True
        break

    exec(
        compile(source, f"{SOURCE_NOTEBOOK.name}::<cell {cell_index}>", "exec"),
        goal1_ns,
        goal1_ns,
    )
    executed_cells += 1

if not runner_definition_seen:
    raise RuntimeError("Could not import Notebook-10 run_goal1 definition boundary.")

required = [
    "RUN_MODE", "N_BOOTSTRAPS", "N_PERMUTATIONS",
    "OUTER_FOLDS", "OUTER_REPEATS", "INNER_FOLDS",
    "BASE_SEED", "OUT", "TABLES", "FIGURES", "OOF_DIR",
    "PRIMARY_MODEL_SPECS", "dx_primary", "sev_primary",
    "dx_all_recordings", "sev_all_60", "sev_90",
    "EXTENDED_SPEC", "QCHAN_TWO_PART_SPEC",
    "AGE_SEX_SPEC", "AGE_SEX_CORE_SPEC",
    "run_or_load_model", "run_sensitivity",
    "bootstrap_model_table", "bootstrap_metric_distribution",
    "bootstrap_ci", "paired_bootstrap_difference",
    "mean_metric_dict", "repeat_metric_table",
    "fit_severity_mean_baseline", "safe_calibration_statistics",
    "final_participant_predictions", "make_wide_primary_oof",
    "descriptive_q_age_table", "generate_figures",
    "split_manifest", "model_checkpoint_paths",
    "safe_read_csv", "atomic_write_csv",
    "observed_paper1_commit", "input_hashes",
]

missing = [name for name in required if name not in goal1_ns]
if missing:
    raise RuntimeError("Notebook-10 import missing required symbols: " + ", ".join(missing))

if goal1_ns["RUN_MODE"] != "FINAL":
    raise RuntimeError(f"Notebook 10 is not in FINAL mode: {goal1_ns['RUN_MODE']!r}")
if int(goal1_ns["N_BOOTSTRAPS"]) != FINAL_BOOTSTRAPS:
    raise RuntimeError("Notebook 10 does not use 2,000 final bootstraps.")
if int(goal1_ns["N_PERMUTATIONS"]) != FINAL_PERMUTATIONS:
    raise RuntimeError("Notebook 10 does not use 1,000 final permutations.")
if int(goal1_ns["OUTER_REPEATS"]) != FINAL_PRIMARY_REPEATS:
    raise RuntimeError("Primary Goal-1 model contract is not 10 outer repeats.")

OUT = Path(goal1_ns["OUT"])
TABLES = Path(goal1_ns["TABLES"])
OOF_DIR = Path(goal1_ns["OOF_DIR"])
CHECKPOINTS = Path(goal1_ns["CHECKPOINTS"])

COMPLETION = OUT / "completion_v1_0"
COMPLETION_TABLES = COMPLETION / "tables"
COMPLETION_AUDIT = COMPLETION / "audit"
FINAL_FIGURES_CANDIDATE = ROOT / "outputs" / "goal1" / "FINAL_FIGURES_CANDIDATE"

for d in [COMPLETION, COMPLETION_TABLES, COMPLETION_AUDIT, FINAL_FIGURES_CANDIDATE]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 76)
print("NOTEBOOK 10 MODELING IMPLEMENTATION IMPORT: PASS")
print("=" * 76)
print("Imported code cells:", executed_cells)
print("Authoritative Goal-1 output tree:", OUT)
print("Primary CV: 5 folds x 10 repeats")
print("Bootstrap replicates:", goal1_ns["N_BOOTSTRAPS"])
print("No permutation recovery/computation cells were executed.")


Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal1-complete-v1.1.0
Run mode: FINAL
Bootstrap replicates: 2000
Permutation replicates: 1000
Permutation CV repeats: 10
Atomic I/O utilities: READY
CANONICAL DATA / QCHAN / SPLIT GATES: PASS
519 recordings | 224 participants | 158 ALS | 66 controls
Primary severity participants: 145
Run signature: cbcec778d60d573f


,model,numeric_n,support_n,binary_n,estimator
0,Age,1,0,0,ridge
1,Support-only,0,3,0,ridge
2,QADD,3,3,0,ridge
3,QGAIN,4,0,0,ridge
4,QREV,1,0,0,ridge
5,QCHAN,4,0,0,ridge
6,Core-Q,12,3,0,ridge
7,Age + Core-Q,13,3,0,ridge
8,Core-Q HGB,12,3,0,hgb


Q REPRESENTATION / MODEL CONTRACT: PASS
FOLD-SAFE QCHAN LEAKAGE SMOKE TEST: PASS
PREDICTOR CONSTRUCTION UNIT TESTS: PASS


,test,task,n_train_participants,n_test_participants,selected_parameter,status
0,Age diagnosis preflight,diagnosis,161,38,1.0000,PASS
1,Support-only diagnosis preflight,diagnosis,179,45,0.0001,PASS
2,Core-Q diagnosis preflight,diagnosis,179,45,0.1000,PASS
3,Core-Q severity preflight,severity,116,29,10.0000,PASS


REAL-DATA MODEL INTEGRATION PREFLIGHT: PASS
NESTED-CV ENGINE: READY
METRICS / BOOTSTRAP / CALIBRATION UTILITIES: READY


,analysis,rows,participants
0,Primary diagnosis index,224,224
1,Primary severity <=60 d,145,145
2,Diagnosis all recordings sensitivity,519,224
3,Severity all <=60 d pairs sensitivity,398,145
4,Severity <=90 d index sensitivity,145,145


PRIMARY / SENSITIVITY POPULATIONS: PASS
CHECKPOINTED MODEL RUNNER: READY
PERMUTATION ENGINE: READY
FIGURE STYLE: READY | font: Arial
NOTEBOOK 10 MODELING IMPLEMENTATION IMPORT: PASS
Imported code cells: 13
Authoritative Goal-1 output tree: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final
Primary CV: 5 folds x 10 repeats
Bootstrap replicates: 2000
No permutation recovery/computation cells were executed.


## 2. Reload/validate the complete primary model ladder

Every model is loaded from its existing validated OOF checkpoint when possible. `run_or_load_model()` recomputes a model only if its checkpoint is absent or invalid.

This stage therefore does not intentionally refit the already-completed model ladder.

In [3]:
PRIMARY_MODEL_SPECS = goal1_ns["PRIMARY_MODEL_SPECS"]
dx_primary = goal1_ns["dx_primary"]
sev_primary = goal1_ns["sev_primary"]

dx_results = OrderedDict()
sev_results = OrderedDict()

for model_name, spec in PRIMARY_MODEL_SPECS.items():
    if model_name == "Core-Q HGB" and not bool(goal1_ns.get("RUN_HGB", True)):
        continue

    dx_results[model_name] = goal1_ns["run_or_load_model"](
        dx_primary,
        model_name,
        spec,
        "diagnosis",
        repeats=FINAL_PRIMARY_REPEATS,
        record_splits=(model_name == "Core-Q"),
    )

    sev_results[model_name] = goal1_ns["run_or_load_model"](
        sev_primary,
        model_name,
        spec,
        "severity",
        repeats=FINAL_PRIMARY_REPEATS,
        record_splits=(model_name == "Core-Q"),
    )

expected_core = {
    "Age", "Support-only", "QADD", "QGAIN", "QREV", "QCHAN",
    "Core-Q", "Age + Core-Q",
}
if not expected_core.issubset(dx_results):
    raise RuntimeError(f"Diagnosis ladder incomplete: {sorted(set(expected_core)-set(dx_results))}")
if not expected_core.issubset(sev_results):
    raise RuntimeError(f"Severity ladder incomplete: {sorted(set(expected_core)-set(sev_results))}")

sev_results_with_baseline = OrderedDict()
sev_results_with_baseline["Mean baseline"] = goal1_ns["fit_severity_mean_baseline"](sev_primary)
sev_results_with_baseline.update(sev_results)

primary_inventory_rows = []
for task, results in [("diagnosis", dx_results), ("severity", sev_results)]:
    for model, oof in results.items():
        primary_inventory_rows.append({
            "task": task,
            "model": model,
            "participants": oof["participant_id"].nunique(),
            "rows": len(oof),
            "repeats": oof["repeat"].nunique(),
            "folds": oof["outer_fold"].nunique(),
            "all_predictions_finite": bool(np.isfinite(pd.to_numeric(oof["prediction"], errors="coerce")).all()),
        })

primary_inventory = pd.DataFrame(primary_inventory_rows)
atomic_csv(primary_inventory, COMPLETION_TABLES / "goal1_primary_oof_inventory.csv")

if not primary_inventory["repeats"].eq(10).all():
    raise RuntimeError("At least one primary model does not contain all 10 OOF repeats.")
if not primary_inventory["folds"].eq(5).all():
    raise RuntimeError("At least one primary model does not contain all 5 outer folds.")
if not primary_inventory["all_predictions_finite"].all():
    raise RuntimeError("At least one primary model contains nonfinite predictions.")

display(primary_inventory)
print("PRIMARY MODEL LADDER OOF GATE: PASS")


Loaded valid checkpoint: diagnosis | Age
Loaded valid checkpoint: severity | Age
Loaded valid checkpoint: diagnosis | Support-only
Loaded valid checkpoint: severity | Support-only
Loaded valid checkpoint: diagnosis | QADD
Loaded valid checkpoint: severity | QADD
Loaded valid checkpoint: diagnosis | QGAIN
Loaded valid checkpoint: severity | QGAIN
Loaded valid checkpoint: diagnosis | QREV
Loaded valid checkpoint: severity | QREV
Loaded valid checkpoint: diagnosis | QCHAN
Loaded valid checkpoint: severity | QCHAN
Loaded valid checkpoint: diagnosis | Core-Q
Loaded valid checkpoint: severity | Core-Q
Loaded valid checkpoint: diagnosis | Age + Core-Q
Loaded valid checkpoint: severity | Age + Core-Q
Loaded valid checkpoint: diagnosis | Core-Q HGB
Loaded valid checkpoint: severity | Core-Q HGB


,task,model,participants,rows,repeats,folds,all_predictions_finite
0,diagnosis,Age,199,1990,10,5,True
1,diagnosis,Support-only,224,2240,10,5,True
2,diagnosis,QADD,224,2240,10,5,True
3,diagnosis,QGAIN,224,2240,10,5,True
4,diagnosis,QREV,224,2240,10,5,True
5,diagnosis,QCHAN,224,2240,10,5,True
6,diagnosis,Core-Q,224,2240,10,5,True
7,diagnosis,Age + Core-Q,199,1990,10,5,True
8,diagnosis,Core-Q HGB,224,2240,10,5,True
9,severity,Age,145,1450,10,5,True


PRIMARY MODEL LADDER OOF GATE: PASS


## 3. Final 2,000-bootstrap primary model uncertainty

Existing bootstrap tables are reused only when they contain the complete expected ladder and explicitly report 2,000 replicates. Otherwise they are regenerated from the authoritative OOF predictions.

In [4]:
def load_valid_bootstrap(path, task, expected_models):
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        frame = pd.read_csv(path)
        required = {
            "model", "task", "metric", "estimate", "ci_low", "ci_high",
            "bootstrap_replicates", "n_participants",
        }
        if not required.issubset(frame.columns):
            return None
        if set(frame["task"].astype(str)) != {task}:
            return None
        if int(pd.to_numeric(frame["bootstrap_replicates"], errors="coerce").min()) != FINAL_BOOTSTRAPS:
            return None
        if not set(expected_models).issubset(set(frame["model"].astype(str))):
            return None
        if not np.isfinite(frame[["estimate", "ci_low", "ci_high"]].to_numpy(float)).all():
            return None
        return frame
    except Exception:
        return None

dx_boot_path = TABLES / "diagnosis_bootstrap_ci.csv"
sev_boot_path = TABLES / "severity_bootstrap_ci.csv"

dx_ci = load_valid_bootstrap(dx_boot_path, "diagnosis", list(dx_results))
if dx_ci is None:
    print("Recomputing diagnosis 2,000-bootstrap table...")
    dx_ci = goal1_ns["bootstrap_model_table"](dx_results, "diagnosis")
    goal1_ns["atomic_write_csv"](dx_ci, dx_boot_path)

sev_ci = load_valid_bootstrap(
    sev_boot_path,
    "severity",
    list(sev_results_with_baseline),
)
if sev_ci is None:
    print("Recomputing severity 2,000-bootstrap table...")
    sev_ci = goal1_ns["bootstrap_model_table"](sev_results_with_baseline, "severity")
    goal1_ns["atomic_write_csv"](sev_ci, sev_boot_path)

# Paired incremental Age + Core-Q vs Age contrast.
dx_delta = goal1_ns["paired_bootstrap_difference"](
    dx_results["Age"], dx_results["Age + Core-Q"],
    "diagnosis", "AUROC", FINAL_BOOTSTRAPS,
    goal1_ns["BASE_SEED"] + 4_000_001,
)
sev_delta = goal1_ns["paired_bootstrap_difference"](
    sev_results["Age"], sev_results["Age + Core-Q"],
    "severity", "MAE", FINAL_BOOTSTRAPS,
    goal1_ns["BASE_SEED"] + 4_000_002,
)
dx_lo, dx_hi = goal1_ns["bootstrap_ci"](dx_delta)
sev_lo, sev_hi = goal1_ns["bootstrap_ci"](sev_delta)

paired_age = pd.DataFrame([
    {
        "task": "diagnosis",
        "contrast": "Age + Core-Q minus Age",
        "metric": "AUROC",
        "estimate": (
            goal1_ns["mean_metric_dict"](dx_results["Age + Core-Q"], "diagnosis")["AUROC"]
            - goal1_ns["mean_metric_dict"](dx_results["Age"], "diagnosis")["AUROC"]
        ),
        "ci_low": dx_lo,
        "ci_high": dx_hi,
        "n_participants": len(
            set(dx_results["Age"]["participant_id"])
            & set(dx_results["Age + Core-Q"]["participant_id"])
        ),
        "bootstrap_replicates": FINAL_BOOTSTRAPS,
    },
    {
        "task": "severity",
        "contrast": "Age + Core-Q minus Age",
        "metric": "MAE",
        "estimate": (
            goal1_ns["mean_metric_dict"](sev_results["Age + Core-Q"], "severity")["MAE"]
            - goal1_ns["mean_metric_dict"](sev_results["Age"], "severity")["MAE"]
        ),
        "ci_low": sev_lo,
        "ci_high": sev_hi,
        "n_participants": len(
            set(sev_results["Age"]["participant_id"])
            & set(sev_results["Age + Core-Q"]["participant_id"])
        ),
        "bootstrap_replicates": FINAL_BOOTSTRAPS,
    },
])
atomic_csv(paired_age, TABLES / "age_incremental_paired_contrast.csv")

if int(dx_ci["bootstrap_replicates"].min()) != 2000:
    raise RuntimeError("Diagnosis bootstrap table is not final 2,000-bootstrap inference.")
if int(sev_ci["bootstrap_replicates"].min()) != 2000:
    raise RuntimeError("Severity bootstrap table is not final 2,000-bootstrap inference.")

print("PRIMARY 2,000-BOOTSTRAP INFERENCE: PASS")
display(dx_ci.loc[dx_ci["metric"].eq("AUROC")])
display(sev_ci.loc[sev_ci["metric"].eq("MAE")])
display(paired_age)


PRIMARY 2,000-BOOTSTRAP INFERENCE: PASS


,model,task,metric,estimate,ci_low,ci_high,bootstrap_replicates,n_participants
0,Age,diagnosis,AUROC,0.740730,0.646480,0.831230,2000,199
3,Support-only,diagnosis,AUROC,0.467376,0.438834,0.495386,2000,224
6,QADD,diagnosis,AUROC,0.707988,0.633786,0.776351,2000,224
9,QGAIN,diagnosis,AUROC,0.687054,0.608876,0.757089,2000,224
12,QREV,diagnosis,AUROC,0.614624,0.533991,0.692021,2000,224
15,QCHAN,diagnosis,AUROC,0.643585,0.573188,0.712165,2000,224
18,Core-Q,diagnosis,AUROC,0.743038,0.675067,0.807226,2000,224
21,Age + Core-Q,diagnosis,AUROC,0.872692,0.820051,0.923585,2000,199
24,Core-Q HGB,diagnosis,AUROC,0.687812,0.618821,0.749439,2000,224


,model,task,metric,estimate,ci_low,ci_high,bootstrap_replicates,n_participants
0,Mean baseline,severity,MAE,2.018414,1.801147,2.245404,2000,145
4,Age,severity,MAE,1.989553,1.764474,2.228496,2000,145
8,Support-only,severity,MAE,1.996109,1.777356,2.216551,2000,145
12,QADD,severity,MAE,2.013920,1.787201,2.238237,2000,145
16,QGAIN,severity,MAE,1.726427,1.531152,1.925909,2000,145
20,QREV,severity,MAE,2.034355,1.824332,2.267010,2000,145
24,QCHAN,severity,MAE,1.931513,1.725125,2.168066,2000,145
28,Core-Q,severity,MAE,1.685235,1.477423,1.898178,2000,145
32,Age + Core-Q,severity,MAE,1.661048,1.466109,1.865268,2000,145
36,Core-Q HGB,severity,MAE,1.685157,1.524977,1.855642,2000,145


,task,contrast,metric,estimate,ci_low,ci_high,n_participants,bootstrap_replicates
0,diagnosis,Age + Core-Q minus Age,AUROC,0.131962,0.055476,0.209315,199,2000
1,severity,Age + Core-Q minus Age,MAE,-0.328506,-0.507410,-0.147314,145,2000


## 4. Hard-audit the final corrected diagnosis permutation null

The invalid earlier diagnosis null that retained the true-label stratified fold assignment is not permitted here.

Notebook 10's corrected diagnosis null must:

- contain exactly permutation IDs 1–1000;
- contain finite AUROCs;
- use the corrected regenerated-stratification design;
- report 3 permutation repeats;
- agree with the independently recomputed first-three-repeat observed Core-Q AUROC;
- agree with the final corrected diagnosis summary JSON;
- report that the old fixed-manifest diagnosis null was not used.

In [5]:
DX_CANONICAL = TABLES / "diagnosis_coreq_permutation_null.csv"
DX_SUMMARY = (
    OUT / "corrected_diagnosis_permutation_v1_1" / "tables"
    / "diagnosis_permutation_final_summary.json"
)

if not DX_CANONICAL.exists() or not DX_SUMMARY.exists():
    raise FileNotFoundError(
        "Corrected diagnosis finalization outputs are missing. "
        "Notebook 10 cell 'FINALIZE CORRECTED DIAGNOSIS PERMUTATION NULL' must have passed."
    )

dx_perm = pd.read_csv(DX_CANONICAL)
dx_summary = json.loads(DX_SUMMARY.read_text(encoding="utf-8"))

required_dx = {
    "permutation", "null_metric", "task", "metric",
    "observed_metric", "empirical_p", "permutation_repeats",
    "permutation_design",
}
if not required_dx.issubset(dx_perm.columns):
    raise RuntimeError(f"Corrected diagnosis table missing: {sorted(required_dx-set(dx_perm.columns))}")

ids = pd.to_numeric(dx_perm["permutation"], errors="raise").astype(int).to_numpy()
if len(dx_perm) != 1000 or dx_perm["permutation"].nunique() != 1000:
    raise RuntimeError("Corrected diagnosis null is not exactly 1,000 unique draws.")
if not np.array_equal(np.sort(ids), np.arange(1, 1001)):
    raise RuntimeError("Corrected diagnosis permutation IDs are not exactly 1..1000.")
if not np.isfinite(pd.to_numeric(dx_perm["null_metric"], errors="coerce")).all():
    raise RuntimeError("Corrected diagnosis null contains nonfinite AUROC.")
if set(dx_perm["task"].astype(str)) != {"diagnosis"}:
    raise RuntimeError("Diagnosis canonical table task mismatch.")
if set(dx_perm["metric"].astype(str)) != {"AUROC"}:
    raise RuntimeError("Diagnosis canonical table metric mismatch.")
if set(pd.to_numeric(dx_perm["permutation_repeats"], errors="raise").astype(int)) != {3}:
    raise RuntimeError("Diagnosis formal null is not the matched 3-repeat implementation.")
if not dx_perm["permutation_design"].astype(str).str.contains(
    "stratification regenerated", case=False, regex=False
).all():
    raise RuntimeError("Diagnosis canonical null does not document regenerated stratification.")

if str(dx_summary.get("status", "")).upper() != "PASS":
    raise RuntimeError("Corrected diagnosis summary is not PASS.")
if bool(dx_summary.get("old_fixed_manifest_diagnosis_null_used", True)):
    raise RuntimeError("Diagnosis final summary indicates the invalid old null was used.")
if int(dx_summary.get("n_permutations", -1)) != 1000:
    raise RuntimeError("Diagnosis summary does not report 1,000 permutations.")
if int(dx_summary.get("permutation_repeats", -1)) != 3:
    raise RuntimeError("Diagnosis summary does not report the matched 3-repeat null.")

dx_observed3 = dx_results["Core-Q"].loc[
    dx_results["Core-Q"]["repeat"].between(1, 3)
].copy()
dx_observed3_auc = float(
    goal1_ns["repeat_metric_table"](dx_observed3, "diagnosis")["AUROC"].mean()
)
dx_primary10_auc = float(
    goal1_ns["repeat_metric_table"](dx_results["Core-Q"], "diagnosis")["AUROC"].mean()
)

if not np.allclose(
    pd.to_numeric(dx_perm["observed_metric"], errors="raise").to_numpy(float),
    dx_observed3_auc,
    atol=1e-12, rtol=0,
):
    raise RuntimeError("Diagnosis matched observed AUROC disagrees with Core-Q OOF repeats 1-3.")

dx_null = pd.to_numeric(dx_perm["null_metric"], errors="raise").to_numpy(float)
dx_extreme = int(np.sum(dx_null >= dx_observed3_auc))
dx_p_recomputed = (1 + dx_extreme) / 1001.0
dx_p_saved = float(pd.to_numeric(dx_perm["empirical_p"], errors="raise").iloc[0])

if not np.isclose(dx_p_recomputed, dx_p_saved, atol=1e-15, rtol=0):
    raise RuntimeError("Saved diagnosis empirical P does not match the recomputed upper-tail P.")

print("=" * 76)
print("CORRECTED DIAGNOSIS PERMUTATION AUDIT: PASS")
print("=" * 76)
print("Primary 10-repeat Core-Q AUROC:", f"{dx_primary10_auc:.6f}")
print("Matched 3-repeat observed AUROC:", f"{dx_observed3_auc:.6f}")
print("Null mean:", f"{np.mean(dx_null):.6f}")
print("Extreme null draws:", dx_extreme)
print("Empirical P:", f"{dx_p_saved:.9f}")


CORRECTED DIAGNOSIS PERMUTATION AUDIT: PASS
Primary 10-repeat Core-Q AUROC: 0.743038
Matched 3-repeat observed AUROC: 0.740410
Null mean: 0.487495
Extreme null draws: 0
Empirical P: 0.000999001


## 5. FINALIZE the newly completed severity permutation null

The severity null is scientifically simpler than diagnosis because severity values did not determine the frozen participant fold allocation. Therefore the first three frozen outer repeats remain fixed after severity-score permutation.

This cell does **not** compute a new permutation. It validates the durable 1,000-row checkpoint and promotes it to the canonical final severity permutation table.

In [6]:
SEV_ROOT = OUT / "severity_permutation_v1_0"
SEV_CHECKPOINT = SEV_ROOT / "severity_permutation_checkpoint.csv"
SEV_CONTRACT = SEV_ROOT / "audit" / "severity_permutation_contract.json"
SEV_TABLE_DIR = SEV_ROOT / "tables"
SEV_CANONICAL = TABLES / "severity_coreq_permutation_null.csv"

if not SEV_CHECKPOINT.exists() or not SEV_CONTRACT.exists():
    raise FileNotFoundError(
        "Severity 1,000-draw checkpoint/contract is missing. "
        "Do not rerun permutations unless the durable checkpoint is genuinely absent."
    )

sev_contract = json.loads(SEV_CONTRACT.read_text(encoding="utf-8"))
sev_null_raw = pd.read_csv(SEV_CHECKPOINT)

required_sev = {
    "permutation", "null_metric", "label_permutation_seed",
    "outer_repeat_seeds", "split_manifest_hash", "engine", "signature",
}
if not required_sev.issubset(sev_null_raw.columns):
    raise RuntimeError(f"Severity checkpoint missing: {sorted(required_sev-set(sev_null_raw.columns))}")

sev_null_raw = (
    sev_null_raw.sort_values("permutation")
    .drop_duplicates("permutation", keep="last")
    .reset_index(drop=True)
)

sev_ids = pd.to_numeric(sev_null_raw["permutation"], errors="raise").astype(int).to_numpy()
if len(sev_null_raw) != 1000 or sev_null_raw["permutation"].nunique() != 1000:
    raise RuntimeError(f"Expected exactly 1,000 unique severity draws; found {len(sev_null_raw)}.")
if not np.array_equal(sev_ids, np.arange(1, 1001)):
    raise RuntimeError("Severity permutation IDs are not exactly ordered 1..1000.")

sev_null = pd.to_numeric(sev_null_raw["null_metric"], errors="coerce").to_numpy(float)
if not np.isfinite(sev_null).all():
    raise RuntimeError("Severity null contains nonfinite MAE.")
if np.any(sev_null < 0):
    raise RuntimeError("Severity null contains impossible negative MAE.")

if str(sev_contract.get("status", "")).upper() != "READY":
    raise RuntimeError(f"Unexpected severity permutation contract status: {sev_contract.get('status')!r}")
if int(sev_contract.get("population_n", -1)) != 145:
    raise RuntimeError("Severity contract population is not 145.")
if int(sev_contract.get("permutation_count", -1)) != 1000:
    raise RuntimeError("Severity contract does not specify 1,000 permutations.")
if int(sev_contract.get("permutation_repeats", -1)) != 3:
    raise RuntimeError("Severity contract does not specify matched 3-repeat inference.")
if bool(sev_contract.get("severity_used_to_construct_split", True)):
    raise RuntimeError("Severity contract incorrectly indicates outcome-dependent outer splitting.")

expected_signature = str(sev_contract["signature"])
if set(sev_null_raw["signature"].astype(str)) != {expected_signature}:
    raise RuntimeError("Severity permutation signature mismatch.")
expected_engine = str(sev_contract["engine"])
if set(sev_null_raw["engine"].astype(str)) != {expected_engine}:
    raise RuntimeError("Severity permutation engine mismatch.")

expected_label_seeds = (
    int(goal1_ns["BASE_SEED"]) + 20_000_000 + np.arange(1, 1001)
)
actual_label_seeds = pd.to_numeric(
    sev_null_raw["label_permutation_seed"], errors="raise"
).astype(int).to_numpy()
if not np.array_equal(actual_label_seeds, expected_label_seeds):
    raise RuntimeError("Severity deterministic label-permutation seeds do not match the frozen rule.")

expected_outer_seed_string = ";".join(
    str(int(goal1_ns["BASE_SEED"]) + r) for r in range(3)
)
if set(sev_null_raw["outer_repeat_seeds"].astype(str)) != {expected_outer_seed_string}:
    raise RuntimeError("Severity outer-repeat seed provenance mismatch.")
if set(sev_null_raw["split_manifest_hash"].astype(str)) != {
    str(sev_contract["split_manifest_sha256"])
}:
    raise RuntimeError("Severity fixed split-manifest hash mismatch.")

sev_observed3 = sev_results["Core-Q"].loc[
    sev_results["Core-Q"]["repeat"].between(1, 3)
].copy()
sev_observed3_mae = float(
    goal1_ns["repeat_metric_table"](sev_observed3, "severity")["MAE"].mean()
)
sev_primary10_mae = float(
    goal1_ns["repeat_metric_table"](sev_results["Core-Q"], "severity")["MAE"].mean()
)

if not np.isclose(
    sev_observed3_mae,
    float(sev_contract["matched_3_repeat_observed_MAE"]),
    atol=1e-12, rtol=0,
):
    raise RuntimeError("Severity matched observed MAE disagrees with the frozen permutation contract.")
if not np.isclose(
    sev_primary10_mae,
    float(sev_contract["primary_10_repeat_MAE"]),
    atol=1e-12, rtol=0,
):
    raise RuntimeError("Severity primary 10-repeat MAE disagrees with the permutation contract.")

# Lower MAE is favorable.
sev_extreme = int(np.sum(sev_null <= sev_observed3_mae))
sev_empirical_p = (1 + sev_extreme) / 1001.0

sev_perm = sev_null_raw.copy()
sev_perm["task"] = "severity"
sev_perm["metric"] = "MAE"
sev_perm["observed_metric"] = sev_observed3_mae
sev_perm["empirical_p"] = sev_empirical_p
sev_perm["permutation_repeats"] = 3
sev_perm["run_mode"] = "FINAL"
sev_perm["permutation_design"] = (
    "participant-level bulbar-score permutation; fixed Phase-0 master outer folds "
    "and severity inner KFold because severity outcome did not construct fold allocation"
)

SEV_TABLE_DIR.mkdir(parents=True, exist_ok=True)
atomic_csv(
    sev_perm,
    SEV_TABLE_DIR / "severity_coreq_permutation_null_final.csv",
)
atomic_csv(sev_perm, SEV_CANONICAL)

sev_summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS",
    "task": "severity",
    "metric": "MAE",
    "primary_10_repeat_MAE": sev_primary10_mae,
    "matched_3_repeat_observed_MAE": sev_observed3_mae,
    "permutation_repeats": 3,
    "n_permutations": 1000,
    "extreme_null_draws": sev_extreme,
    "empirical_p": sev_empirical_p,
    "null_mean": float(np.mean(sev_null)),
    "null_sd": float(np.std(sev_null, ddof=1)),
    "null_median": float(np.median(sev_null)),
    "null_q025": float(np.quantile(sev_null, 0.025)),
    "null_q975": float(np.quantile(sev_null, 0.975)),
    "engine": expected_engine,
    "signature": expected_signature,
    "fixed_master_outer_folds": True,
    "severity_used_to_construct_split": False,
    "fold_safe_QCHAN_recomputed": True,
    "preprocessing_and_tuning_recomputed": True,
}
atomic_json(sev_summary, SEV_TABLE_DIR / "severity_permutation_final_summary.json")

print("=" * 76)
print("SEVERITY PERMUTATION NULL: FINAL PASS")
print("=" * 76)
print("Permutations:", len(sev_null))
print("Primary 10-repeat Core-Q MAE:", f"{sev_primary10_mae:.6f}")
print("Matched 3-repeat observed MAE:", f"{sev_observed3_mae:.6f}")
print("Null mean:", f"{np.mean(sev_null):.6f}")
print("Null SD:", f"{np.std(sev_null, ddof=1):.6f}")
print("95% null interval:", f"[{np.quantile(sev_null, .025):.6f}, {np.quantile(sev_null, .975):.6f}]")
print("Extreme null draws:", sev_extreme)
print("Empirical P:", f"{sev_empirical_p:.9f}")


SEVERITY PERMUTATION NULL: FINAL PASS
Permutations: 1000
Primary 10-repeat Core-Q MAE: 1.685235
Matched 3-repeat observed MAE: 1.680926
Null mean: 2.031577
Null SD: 0.022963
95% null interval: [1.979323, 2.077298]
Extreme null draws: 0
Empirical P: 0.000999001


## 6. Write the authoritative combined permutation artifact and implementation amendment

The original Notebook-10 all-in-one seal expected 10 permutation CV repeats. The corrected final implementation instead uses matched three-repeat statistics for both tasks while preserving the primary 10-repeat model estimates.

This discrepancy must be visible in the machine-readable record and later in the Methods.

In [7]:
goal1_permutation = pd.concat([dx_perm, sev_perm], ignore_index=True, sort=False)
atomic_csv(goal1_permutation, TABLES / "goal1_permutation.csv")

permutation_method_amendment = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "FINAL_IMPLEMENTATION_AMENDMENT",
    "primary_model_estimation": {
        "outer_folds": 5,
        "outer_repeats": 10,
        "inner_folds": 5,
    },
    "formal_permutation_inference": {
        "n_permutations_per_task": 1000,
        "outer_folds": 5,
        "matched_outer_repeats": 3,
        "observed_and_null_statistics_use_identical_repeat_subset": True,
    },
    "diagnosis": {
        "metric": "AUROC",
        "tail": "upper",
        "reason_for_corrected_engine": (
            "The fixed true-label stratified fold manifest produced an invalid "
            "permutation null because class labels had generated the diagnosis "
            "stratification. Final nulls regenerate outer and inner diagnosis "
            "stratification after each participant-level label permutation."
        ),
        "old_fixed_manifest_null_used": False,
    },
    "severity": {
        "metric": "MAE",
        "tail": "lower",
        "outer_fold_rule": (
            "The frozen master folds remain fixed because the continuous severity "
            "outcome did not construct the participant fold assignment."
        ),
    },
    "reporting_requirement": (
        "Manuscript Methods must distinguish 10-repeat primary performance estimation "
        "from matched 3-repeat formal permutation inference."
    ),
}
atomic_json(
    permutation_method_amendment,
    COMPLETION_AUDIT / "goal1_permutation_method_amendment.json",
)

perm_summary = pd.DataFrame([
    {
        "task": "diagnosis",
        "primary_10_repeat_metric": dx_primary10_auc,
        "matched_3_repeat_observed": dx_observed3_auc,
        "null_mean": float(np.mean(dx_null)),
        "null_sd": float(np.std(dx_null, ddof=1)),
        "extreme_null_draws": dx_extreme,
        "empirical_p": dx_p_saved,
        "B": 1000,
        "favorable_tail": "upper",
    },
    {
        "task": "severity",
        "primary_10_repeat_metric": sev_primary10_mae,
        "matched_3_repeat_observed": sev_observed3_mae,
        "null_mean": float(np.mean(sev_null)),
        "null_sd": float(np.std(sev_null, ddof=1)),
        "extreme_null_draws": sev_extreme,
        "empirical_p": sev_empirical_p,
        "B": 1000,
        "favorable_tail": "lower",
    },
])
atomic_csv(perm_summary, COMPLETION_TABLES / "goal1_permutation_final_summary.csv")
display(perm_summary)

print("COMBINED GOAL-1 PERMUTATION ARTIFACT + METHOD AMENDMENT: PASS")


,task,primary_10_repeat_metric,matched_3_repeat_observed,null_mean,null_sd,extreme_null_draws,empirical_p,B,favorable_tail
0,diagnosis,0.743038,0.740410,0.487495,0.038132,0,0.000999,1000,upper
1,severity,1.685235,1.680926,2.031577,0.022963,0,0.000999,1000,lower


COMBINED GOAL-1 PERMUTATION ARTIFACT + METHOD AMENDMENT: PASS


## 7. Run/resume the complete Goal-1 sensitivity package

This reproduces Notebook 10's exact sensitivity specification.

Any valid existing sensitivity OOF checkpoint is reused. Only a missing/invalid sensitivity model is recomputed.

All displayed sensitivity CIs are then recomputed with **2,000 participant-cluster bootstrap replicates**.

In [8]:
sensitivity_results = []
sensitivity_oof = {}

def add_sensitivity(label, task, oof, metric):
    observed = goal1_ns["mean_metric_dict"](oof, task)[metric]
    dist = goal1_ns["bootstrap_metric_distribution"](
        oof,
        task,
        metric,
        FINAL_BOOTSTRAPS,
        int(goal1_ns["BASE_SEED"]) + 8_000_000 + len(sensitivity_results),
    )
    lo, hi = goal1_ns["bootstrap_ci"](dist)

    sensitivity_results.append({
        "task": task,
        "sensitivity": label,
        "metric": metric,
        "estimate": observed,
        "ci_low": lo,
        "ci_high": hi,
        "n_participants": oof["participant_id"].nunique(),
        "n_rows_per_repeat": len(oof.loc[oof["repeat"].eq(1)]),
        "bootstrap_replicates": FINAL_BOOTSTRAPS,
    })
    sensitivity_oof[(task, label)] = oof

# Primary references.
add_sensitivity("Primary index Core-Q", "diagnosis", dx_results["Core-Q"], "AUROC")
add_sensitivity("Primary ≤60 d Core-Q", "severity", sev_results["Core-Q"], "MAE")

# Repeated-recording / repeated-pair analyses with participant-normalized contribution.
add_sensitivity(
    "All recordings, participant-weighted",
    "diagnosis",
    goal1_ns["run_sensitivity"](
        "Sensitivity all recordings Core-Q",
        goal1_ns["dx_all_recordings"],
        PRIMARY_MODEL_SPECS["Core-Q"],
        "diagnosis",
    ),
    "AUROC",
)
add_sensitivity(
    "All ≤60 d pairs, participant-weighted",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity all 60d pairs Core-Q",
        goal1_ns["sev_all_60"],
        PRIMARY_MODEL_SPECS["Core-Q"],
        "severity",
    ),
    "MAE",
)

# Extended-Q.
add_sensitivity(
    "Extended-Q",
    "diagnosis",
    goal1_ns["run_sensitivity"](
        "Sensitivity Extended-Q",
        dx_primary,
        goal1_ns["EXTENDED_SPEC"],
        "diagnosis",
    ),
    "AUROC",
)
add_sensitivity(
    "Extended-Q",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity Extended-Q",
        sev_primary,
        goal1_ns["EXTENDED_SPEC"],
        "severity",
    ),
    "MAE",
)

# QCHAN two-part representation.
add_sensitivity(
    "QCHAN two-part Core-Q",
    "diagnosis",
    goal1_ns["run_sensitivity"](
        "Sensitivity QCHAN two-part",
        dx_primary,
        goal1_ns["QCHAN_TWO_PART_SPEC"],
        "diagnosis",
    ),
    "AUROC",
)
add_sensitivity(
    "QCHAN two-part Core-Q",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity QCHAN two-part",
        sev_primary,
        goal1_ns["QCHAN_TWO_PART_SPEC"],
        "severity",
    ),
    "MAE",
)

# Expanded clinical pairing window.
add_sensitivity(
    "≤90 d index Core-Q",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity 90d Core-Q",
        goal1_ns["sev_90"],
        PRIMARY_MODEL_SPECS["Core-Q"],
        "severity",
    ),
    "MAE",
)

# Complete-case age + sex benchmarks.
add_sensitivity(
    "Age + sex complete-case",
    "diagnosis",
    goal1_ns["run_sensitivity"](
        "Sensitivity Age + sex",
        dx_primary,
        goal1_ns["AGE_SEX_SPEC"],
        "diagnosis",
    ),
    "AUROC",
)
add_sensitivity(
    "Age + sex + Core-Q complete-case",
    "diagnosis",
    goal1_ns["run_sensitivity"](
        "Sensitivity Age + sex + Core-Q",
        dx_primary,
        goal1_ns["AGE_SEX_CORE_SPEC"],
        "diagnosis",
    ),
    "AUROC",
)
add_sensitivity(
    "Age + sex complete-case",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity Age + sex",
        sev_primary,
        goal1_ns["AGE_SEX_SPEC"],
        "severity",
    ),
    "MAE",
)
add_sensitivity(
    "Age + sex + Core-Q complete-case",
    "severity",
    goal1_ns["run_sensitivity"](
        "Sensitivity Age + sex + Core-Q",
        sev_primary,
        goal1_ns["AGE_SEX_CORE_SPEC"],
        "severity",
    ),
    "MAE",
)

# Nonlinear sensitivity already in the model ladder.
if "Core-Q HGB" in dx_results:
    add_sensitivity("Core-Q HGB", "diagnosis", dx_results["Core-Q HGB"], "AUROC")
if "Core-Q HGB" in sev_results:
    add_sensitivity("Core-Q HGB", "severity", sev_results["Core-Q HGB"], "MAE")

sensitivity_table = pd.DataFrame(sensitivity_results)

EXPECTED_SENSITIVITY_LABELS = {
    ("diagnosis", "Primary index Core-Q"),
    ("diagnosis", "All recordings, participant-weighted"),
    ("diagnosis", "Extended-Q"),
    ("diagnosis", "QCHAN two-part Core-Q"),
    ("diagnosis", "Age + sex complete-case"),
    ("diagnosis", "Age + sex + Core-Q complete-case"),
    ("severity", "Primary ≤60 d Core-Q"),
    ("severity", "All ≤60 d pairs, participant-weighted"),
    ("severity", "Extended-Q"),
    ("severity", "QCHAN two-part Core-Q"),
    ("severity", "≤90 d index Core-Q"),
    ("severity", "Age + sex complete-case"),
    ("severity", "Age + sex + Core-Q complete-case"),
}
if "Core-Q HGB" in dx_results:
    EXPECTED_SENSITIVITY_LABELS.add(("diagnosis", "Core-Q HGB"))
if "Core-Q HGB" in sev_results:
    EXPECTED_SENSITIVITY_LABELS.add(("severity", "Core-Q HGB"))

observed_labels = set(
    zip(sensitivity_table["task"].astype(str), sensitivity_table["sensitivity"].astype(str))
)
if observed_labels != EXPECTED_SENSITIVITY_LABELS:
    raise RuntimeError(
        "Goal-1 sensitivity set is incomplete or unexpected.\n"
        f"Missing: {sorted(EXPECTED_SENSITIVITY_LABELS-observed_labels)}\n"
        f"Extra: {sorted(observed_labels-EXPECTED_SENSITIVITY_LABELS)}"
    )
if not sensitivity_table["bootstrap_replicates"].eq(2000).all():
    raise RuntimeError("Sensitivity CI package is not uniformly 2,000-bootstrap.")
if not np.isfinite(
    sensitivity_table[["estimate", "ci_low", "ci_high"]].to_numpy(float)
).all():
    raise RuntimeError("Sensitivity summary contains nonfinite primary estimates/CIs.")

atomic_csv(sensitivity_table, TABLES / "goal1_sensitivity_summary.csv")
atomic_csv(sensitivity_table, COMPLETION_TABLES / "goal1_sensitivity_summary_FINAL.csv")

print("=" * 76)
print("GOAL 1 PRESPECIFIED SENSITIVITY PACKAGE: PASS")
print("=" * 76)
print("Sensitivity rows:", len(sensitivity_table))
display(sensitivity_table)


diagnosis | Sensitivity all recordings Core-Q | repeat 1/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 2/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 3/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 4/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 5/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 6/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 7/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 8/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 9/10 complete
diagnosis | Sensitivity all recordings Core-Q | repeat 10/10 complete
severity  | Sensitivity all 60d pairs Core-Q | repeat 1/10 complete
severity  | Sensitivity all 60d pairs Core-Q | repeat 2/10 complete
severity  | Sensitivity all 60d pairs Core-Q | repeat 3/10 complete
severity  | Sensitivity all 60d pairs Core-Q | repeat 4/10 complete
severity  | Sensitivity all 60d pairs

,task,sensitivity,metric,estimate,ci_low,ci_high,n_participants,n_rows_per_repeat,bootstrap_replicates
0,diagnosis,Primary index Core-Q,AUROC,0.743038,0.673665,0.805439,224,224,2000
1,severity,Primary ≤60 d Core-Q,MAE,1.685235,1.481838,1.895487,145,145,2000
2,diagnosis,"All recordings, participant-weighted",AUROC,0.732144,0.666204,0.796309,224,519,2000
3,severity,"All ≤60 d pairs, participant-weighted",MAE,1.724780,1.540659,1.924512,145,398,2000
4,diagnosis,Extended-Q,AUROC,0.751707,0.684169,0.810570,224,224,2000
5,severity,Extended-Q,MAE,1.625722,1.438950,1.835867,145,145,2000
6,diagnosis,QCHAN two-part Core-Q,AUROC,0.743124,0.676154,0.803161,224,224,2000
7,severity,QCHAN two-part Core-Q,MAE,1.688959,1.499666,1.896343,145,145,2000
8,severity,≤90 d index Core-Q,MAE,1.685235,1.490716,1.890606,145,145,2000
9,diagnosis,Age + sex complete-case,AUROC,0.734903,0.638360,0.826485,199,199,2000


## 8. Calibration, descriptive Q–age check, and canonical Goal-1 artifacts

This cell rebuilds the remaining low-cost reporting tables from the authoritative OOF state. No predictive model is changed.

In [9]:
# Native diagnosis calibration.
calibration_rows = []
calibration_curve_rows = []

for model_name in ["Age", "Core-Q", "Age + Core-Q"]:
    oof = dx_results[model_name]
    calibration_rows.append({
        "model": model_name,
        "n_participants": oof["participant_id"].nunique(),
        **goal1_ns["safe_calibration_statistics"](oof),
    })

    participant = goal1_ns["final_participant_predictions"](oof, "diagnosis")
    smooth = goal1_ns["lowess"](
        participant["y"].to_numpy(float),
        participant["prediction"].to_numpy(float),
        frac=0.40,
        it=0,
        return_sorted=True,
    )
    for x, yhat in smooth:
        calibration_curve_rows.append({
            "model": model_name,
            "predicted_probability": float(x),
            "observed_fraction_smooth": float(np.clip(yhat, 0, 1)),
        })

calibration_table = pd.DataFrame(calibration_rows)
calibration_curves = pd.DataFrame(calibration_curve_rows)
atomic_csv(calibration_table, TABLES / "diagnosis_calibration_statistics.csv")
atomic_csv(calibration_curves, TABLES / "diagnosis_calibration_curve.csv")

# Descriptive only.
q_age = goal1_ns["descriptive_q_age_table"]()
atomic_csv(q_age, TABLES / "descriptive_q_age_associations.csv")

# Canonical participant-level OOF files.
goal1_dx_oof = goal1_ns["make_wide_primary_oof"](dx_results, "diagnosis")
goal1_bulbar_oof = goal1_ns["make_wide_primary_oof"](sev_results, "severity")
atomic_csv(goal1_dx_oof, TABLES / "goal1_dx_oof.csv")
atomic_csv(goal1_bulbar_oof, TABLES / "goal1_bulbar_oof.csv")

goal1_metrics = pd.concat([dx_ci, sev_ci], ignore_index=True, sort=False)
atomic_csv(goal1_metrics, TABLES / "goal1_metrics.csv")

# Split provenance: master outer manifest + recorded Core-Q inner split logs.
split_frames = []
outer_copy = goal1_ns["split_manifest"].copy()
outer_copy["task"] = "master"
outer_copy["model"] = "master"
outer_copy["inner_fold"] = 0
outer_copy["role"] = "outer_assignment"
split_frames.append(
    outer_copy[
        [
            "task", "model", "repeat", "outer_fold",
            "inner_fold", "participant_id", "role",
        ]
    ]
)

for task in ["diagnosis", "severity"]:
    split_path = goal1_ns["model_checkpoint_paths"](task, "Core-Q")["splits"]
    if not split_path.exists():
        raise FileNotFoundError(f"Core-Q split log missing: {split_path}")
    split_frames.append(pd.read_csv(split_path))

goal1_splits = pd.concat(split_frames, ignore_index=True, sort=False)
atomic_csv(goal1_splits, TABLES / "goal1_splits.csv")

canonical = {
    "goal1_dx_oof.csv": goal1_dx_oof,
    "goal1_bulbar_oof.csv": goal1_bulbar_oof,
    "goal1_metrics.csv": goal1_metrics,
    "goal1_permutation.csv": goal1_permutation,
    "goal1_splits.csv": goal1_splits,
    "goal1_sensitivity_summary.csv": sensitivity_table,
}

canonical_audit = pd.DataFrame([
    {
        "artifact": name,
        "rows": len(frame),
        "columns": len(frame.columns),
        "empty": frame.empty,
    }
    for name, frame in canonical.items()
])
atomic_csv(canonical_audit, COMPLETION_TABLES / "goal1_canonical_artifact_audit.csv")

if canonical_audit["empty"].any():
    raise RuntimeError("At least one canonical Goal-1 output is empty.")

display(canonical_audit)
display(calibration_table)
print("CANONICAL GOAL-1 REPORTING TABLES: PASS")


,artifact,rows,columns,empty
0,goal1_dx_oof.csv,1966,14,False
1,goal1_bulbar_oof.csv,1305,14,False
2,goal1_metrics.csv,67,8,False
3,goal1_permutation.csv,2000,14,False
4,goal1_splits.csv,20690,8,False
5,goal1_sensitivity_summary.csv,15,9,False


,model,n_participants,calibration_intercept,calibration_slope,joint_calibration_intercept,calibration_status
0,Age,199,0.000863,0.999209,0.001794,ok
1,Core-Q,224,0.014490,1.001242,0.013605,ok
2,Age + Core-Q,199,0.022104,1.208768,-0.147749,ok


CANONICAL GOAL-1 REPORTING TABLES: PASS


## 9. Generate Figure 2 and supplementary figure candidates

All candidate publication figures are written to:

`outputs/goal1/FINAL_FIGURES_CANDIDATE/`

The numerical source tables remain in the governed Goal-1 final output tree.

These figures are **candidates for visual review**, not yet the publication freeze.

In [10]:
# Redirect only the figure export folder in Notebook-10's defining namespace.
goal1_ns["FIGURES"] = FINAL_FIGURES_CANDIDATE
goal1_ns["TABLES"] = TABLES

goal1_ns["generate_figures"](
    dx_results,
    sev_results,
    dx_ci,
    sev_ci,
    dx_perm,
    sev_perm,
    sensitivity_table,
    calibration_curves,
    q_age,
)

expected_figure_stems = [
    "Figure2_Goal1_information_availability_final",
    "FigureS1_cohort_flow_final",
    "FigureS2_CoreQ_availability_final",
    "FigureS3_diagnosis_calibration_final",
    "FigureS4_bulbar_prediction_residuals_final",
    "FigureS5_Goal1_sensitivities_final",
]

figure_rows = []
for stem in expected_figure_stems:
    for suffix in [".pdf", ".svg", ".png"]:
        path = FINAL_FIGURES_CANDIDATE / f"{stem}{suffix}"
        figure_rows.append({
            "stem": stem,
            "suffix": suffix,
            "path": str(path),
            "exists": path.exists(),
            "bytes": path.stat().st_size if path.exists() else 0,
        })

figure_inventory = pd.DataFrame(figure_rows)
atomic_csv(figure_inventory, COMPLETION_TABLES / "goal1_candidate_figure_inventory.csv")

if not figure_inventory["exists"].all():
    display(figure_inventory.loc[~figure_inventory["exists"]])
    raise RuntimeError("At least one required Goal-1 candidate figure was not written.")
if not figure_inventory["bytes"].gt(0).all():
    raise RuntimeError("At least one Goal-1 candidate figure is empty.")

display(figure_inventory)
print("=" * 76)
print("GOAL 1 FIGURE CANDIDATE PACKAGE: WRITTEN")
print("=" * 76)
print("Folder:", FINAL_FIGURES_CANDIDATE)


,stem,suffix,path,exists,bytes
0,Figure2_Goal1_information_availability_final,.pdf,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,49628
1,Figure2_Goal1_information_availability_final,.svg,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,55780
2,Figure2_Goal1_information_availability_final,.png,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,439011
3,FigureS1_cohort_flow_final,.pdf,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,28120
4,FigureS1_cohort_flow_final,.svg,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,7399
5,FigureS1_cohort_flow_final,.png,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,130288
6,FigureS2_CoreQ_availability_final,.pdf,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,26103
7,FigureS2_CoreQ_availability_final,.svg,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,9009
8,FigureS2_CoreQ_availability_final,.png,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,300676
9,FigureS3_diagnosis_calibration_final,.pdf,C:\Users\musikicn\Desktop\Nevena_project\Paper...,True,28807


GOAL 1 FIGURE CANDIDATE PACKAGE: WRITTEN
Folder: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\FINAL_FIGURES_CANDIDATE


## 10. Computational completion seal — pending visual figure review

Passing this cell means the statistical/computational Goal-1 package is complete.

The only remaining Goal-1 step is a visual/scientific review of the candidate figures, followed by a tiny final publication-figure/freeze notebook.

In [11]:
required_artifacts = [
    TABLES / "goal1_dx_oof.csv",
    TABLES / "goal1_bulbar_oof.csv",
    TABLES / "goal1_metrics.csv",
    TABLES / "goal1_permutation.csv",
    TABLES / "goal1_splits.csv",
    TABLES / "goal1_sensitivity_summary.csv",
    TABLES / "diagnosis_coreq_permutation_null.csv",
    TABLES / "severity_coreq_permutation_null.csv",
    TABLES / "diagnosis_bootstrap_ci.csv",
    TABLES / "severity_bootstrap_ci.csv",
    TABLES / "diagnosis_calibration_statistics.csv",
    TABLES / "diagnosis_calibration_curve.csv",
    COMPLETION_AUDIT / "goal1_permutation_method_amendment.json",
    COMPLETION_TABLES / "goal1_permutation_final_summary.csv",
    COMPLETION_TABLES / "goal1_canonical_artifact_audit.csv",
    COMPLETION_TABLES / "goal1_candidate_figure_inventory.csv",
]

for stem in expected_figure_stems:
    required_artifacts.extend([
        FINAL_FIGURES_CANDIDATE / f"{stem}.pdf",
        FINAL_FIGURES_CANDIDATE / f"{stem}.svg",
        FINAL_FIGURES_CANDIDATE / f"{stem}.png",
    ])

missing = [
    str(p) for p in required_artifacts
    if not p.exists() or p.stat().st_size == 0
]
if missing:
    raise RuntimeError(
        "Cannot write Goal-1 completion seal; missing/empty artifacts:\n"
        + "\n".join(missing)
    )

artifact_hashes = {
    str(p.relative_to(ROOT)): sha256_file(p)
    for p in required_artifacts
}

seal = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "status": "PASS_PENDING_FIGURE_REVIEW",
    "goal": 1,
    "engine": COMPLETION_ENGINE,
    "source_notebook": str(SOURCE_NOTEBOOK.relative_to(ROOT)),
    "source_notebook_sha256": SOURCE_NOTEBOOK_SHA256,
    "primary_outer_folds": 5,
    "primary_outer_repeats": 10,
    "inner_folds": 5,
    "bootstrap_replicates": 2000,
    "formal_permutations_per_task": 1000,
    "formal_permutation_repeats": 3,
    "diagnosis_permutation_design": (
        "participant-level label permutation with outer and inner diagnosis "
        "stratification regenerated after every shuffle"
    ),
    "severity_permutation_design": (
        "participant-level bulbar-score permutation with fixed outcome-independent "
        "master outer folds and severity inner KFold"
    ),
    "old_invalid_fixed_manifest_diagnosis_null_used": False,
    "sensitivity_rows": int(len(sensitivity_table)),
    "main_figure_candidate": "Figure2_Goal1_information_availability_final",
    "supplementary_figure_candidates": expected_figure_stems[1:],
    "interpretation_boundary": (
        "Above-null Q-only prediction establishes reproducible clinically structured "
        "information in the recording-quality representation. It does not establish "
        "technical origin, confounding, spuriousness, shortcut learning, or causal "
        "acquisition effects."
    ),
    "next_step": (
        "Visually audit Figure 2 and supplementary candidate PNGs; then create the "
        "final Goal-1 publication figure/freeze seal without changing statistical results."
    ),
    "artifact_hashes": artifact_hashes,
}

atomic_json(seal, COMPLETION / "GOAL1_COMPLETION_MANIFEST.json")
atomic_json(
    {
        "status": "PASS_PENDING_FIGURE_REVIEW",
        "manifest": str((COMPLETION / "GOAL1_COMPLETION_MANIFEST.json").relative_to(ROOT)),
    },
    COMPLETION / "READY_FOR_FINAL_FIGURE_REVIEW.json",
)

print("=" * 78)
print("GOAL 1 COMPUTATIONAL COMPLETION: PASS_PENDING_FIGURE_REVIEW")
print("=" * 78)
print("Manifest:", COMPLETION / "GOAL1_COMPLETION_MANIFEST.json")
print("Candidate figures:", FINAL_FIGURES_CANDIDATE)
print()
print("PRIMARY GOAL-1 RESULTS FOR REVIEW")
display(
    dx_ci.loc[
        dx_ci["metric"].eq("AUROC"),
        ["model", "n_participants", "estimate", "ci_low", "ci_high"],
    ]
)
display(
    sev_ci.loc[
        sev_ci["metric"].eq("MAE"),
        ["model", "n_participants", "estimate", "ci_low", "ci_high"],
    ]
)
display(perm_summary)
display(sensitivity_table)
print()
print("NEXT: upload the six candidate PNG figures for final visual/scientific audit.")


GOAL 1 COMPUTATIONAL COMPLETION: PASS_PENDING_FIGURE_REVIEW
Manifest: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\goal1_complete_v1_1\final\completion_v1_0\GOAL1_COMPLETION_MANIFEST.json
Candidate figures: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal1\FINAL_FIGURES_CANDIDATE

PRIMARY GOAL-1 RESULTS FOR REVIEW


,model,n_participants,estimate,ci_low,ci_high
0,Age,199,0.740730,0.646480,0.831230
3,Support-only,224,0.467376,0.438834,0.495386
6,QADD,224,0.707988,0.633786,0.776351
9,QGAIN,224,0.687054,0.608876,0.757089
12,QREV,224,0.614624,0.533991,0.692021
15,QCHAN,224,0.643585,0.573188,0.712165
18,Core-Q,224,0.743038,0.675067,0.807226
21,Age + Core-Q,199,0.872692,0.820051,0.923585
24,Core-Q HGB,224,0.687812,0.618821,0.749439


,model,n_participants,estimate,ci_low,ci_high
0,Mean baseline,145,2.018414,1.801147,2.245404
4,Age,145,1.989553,1.764474,2.228496
8,Support-only,145,1.996109,1.777356,2.216551
12,QADD,145,2.013920,1.787201,2.238237
16,QGAIN,145,1.726427,1.531152,1.925909
20,QREV,145,2.034355,1.824332,2.267010
24,QCHAN,145,1.931513,1.725125,2.168066
28,Core-Q,145,1.685235,1.477423,1.898178
32,Age + Core-Q,145,1.661048,1.466109,1.865268
36,Core-Q HGB,145,1.685157,1.524977,1.855642


,task,primary_10_repeat_metric,matched_3_repeat_observed,null_mean,null_sd,extreme_null_draws,empirical_p,B,favorable_tail
0,diagnosis,0.743038,0.740410,0.487495,0.038132,0,0.000999,1000,upper
1,severity,1.685235,1.680926,2.031577,0.022963,0,0.000999,1000,lower


,task,sensitivity,metric,estimate,ci_low,ci_high,n_participants,n_rows_per_repeat,bootstrap_replicates
0,diagnosis,Primary index Core-Q,AUROC,0.743038,0.673665,0.805439,224,224,2000
1,severity,Primary ≤60 d Core-Q,MAE,1.685235,1.481838,1.895487,145,145,2000
2,diagnosis,"All recordings, participant-weighted",AUROC,0.732144,0.666204,0.796309,224,519,2000
3,severity,"All ≤60 d pairs, participant-weighted",MAE,1.724780,1.540659,1.924512,145,398,2000
4,diagnosis,Extended-Q,AUROC,0.751707,0.684169,0.810570,224,224,2000
5,severity,Extended-Q,MAE,1.625722,1.438950,1.835867,145,145,2000
6,diagnosis,QCHAN two-part Core-Q,AUROC,0.743124,0.676154,0.803161,224,224,2000
7,severity,QCHAN two-part Core-Q,MAE,1.688959,1.499666,1.896343,145,145,2000
8,severity,≤90 d index Core-Q,MAE,1.685235,1.490716,1.890606,145,145,2000
9,diagnosis,Age + sex complete-case,AUROC,0.734903,0.638360,0.826485,199,199,2000



NEXT: upload the six candidate PNG figures for final visual/scientific audit.
